# Transformer Univariate


In this section we implement the Transformer Model using the **TimeSeriesDataset** approach with one-hot encoding.

The Transformer Forecaster is a self-attention-based neural network designed for time series forecasting. It uses one-hot encoding to identify individual series (1502 unique series), processing one series at a time. Unlike recurrent models that process sequentially, the Transformer uses attention mechanisms to capture dependencies across the entire sequence simultaneously.

Key Insight: Each training sample represents a single series with its one-hot encoded identifier, allowing the model to learn series-specific patterns while leveraging parallel processing through self-attention mechanisms.

## Architecture

```bash
Input (seq_length, input_size)
    ↓
Input Projection (input_size → d_model)
    ↓
Positional Encoding
    ↓
Transformer Encoder Layers (2 layers)
  ├─ Multi-Head Self-Attention (4 heads)
  ├─ Add & Norm
  ├─ Feedforward (d_model → 256 → d_model)
  └─ Add & Norm
    ↓
Global Average Pooling
    ↓
Dropout
    ↓
Fully Connected (d_model → 1)
    ↓
Output (1 prediction)
```

## Layer Breakdown

- **Input Projection:** Linear layer mapping input features to model dimension (d_model=64)
- **Positional Encoding:** Adds positional information to preserve temporal order
- **Transformer Encoder:** 2 stacked encoder layers with multi-head self-attention (4 heads)
- **Feedforward Network:** 256-dimensional feedforward network within each encoder layer
- **Global Average Pooling:** Aggregates information across all timesteps
- **Dropout:** Applied before final output
- **Output Layer:** Single fully connected layer producing 1-step forecast

## Advantages

- **Parallel Processing:** Processes entire sequence at once (much faster than RNN/LSTM/GRU)
- **Long-Range Dependencies:** Self-attention captures relationships between any two timesteps directly
- **No Gradient Vanishing:** No recurrent connections means no vanishing gradient problem
- **Interpretable:** Attention weights show which timesteps the model focuses on

## Limitations

- **More Parameters:** Requires more data than RNN/LSTM to train effectively
- **Memory Intensive:** Attention mechanism has O(n²) memory complexity with sequence length
- **Positional Encoding Required:** Needs explicit encoding to understand temporal order
- **Short Sequences:** May be overkill for very short sequences (<10 timesteps)

## When to Use

- Dataset is large (>10,000 samples)
- Sequences are moderately long (10-50 timesteps)
- Need to capture complex long-range dependencies
- Training speed is important (parallelizable)
- Interpretability of attention patterns is valuable

## Key Hyperparameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| d_model | 64 | Embedding dimension - higher captures more complexity |
| nhead | 4 | Number of attention heads - allows attending to different aspects |
| num_layers | 2 | Depth of transformer - more layers = more capacity |
| dim_feedforward | 256 | Size of feedforward network within each layer |
| dropout | 0.2 | Dropout rate for regularization |

## Model

In [ ]:
import torch 
import torch.nn as nn

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Positional encoding for Transformer model.
    Adds information about the position of tokens in the sequence.
    """
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # Shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, d_model)
        return x + self.pe[:, :x.size(1), :]


class TransformerForecaster(nn.Module):
    """
    Transformer model for MULTIVARIATE time series forecasting.
    Architecture: 
        Input Projection -> Positional Encoding -> 
        Transformer Encoder -> Global Average Pooling -> 
        Dropout -> Fully Connected
    
    Uses self-attention mechanism to capture dependencies.
    Can process entire sequence in parallel (unlike RNN/LSTM).
    """
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, 
                 dim_feedforward=256, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            d_model: Dimension of the model (must be divisible by nhead)
            nhead: Number of attention heads
            num_layers: Number of transformer encoder layers
            dim_feedforward: Dimension of feedforward network
            dropout: Dropout rate
        """
        super(TransformerForecaster, self).__init__()
        
        self.input_size = input_size
        self.d_model = d_model
        
        # Input projection: map input_size to d_model
        self.input_projection = nn.Linear(input_size, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # Important: batch_first=True for (batch, seq, feature) format
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Output layer
        self.fc = nn.Linear(d_model, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # Project input to d_model dimensions
        x = self.input_projection(x)  # (batch_size, seq_length, d_model)
        
        # Add positional encoding
        x = self.pos_encoder(x)  # (batch_size, seq_length, d_model)
        
        # Pass through transformer encoder
        transformer_out = self.transformer_encoder(x)  # (batch_size, seq_length, d_model)
        
        # Global average pooling over sequence dimension
        # Alternative: use last token or first token (like BERT's [CLS])
        pooled = transformer_out.mean(dim=1)  # (batch_size, d_model)
        
        # Apply dropout
        out = self.dropout(pooled)
        
        # Fully connected layer
        out = self.fc(out)  # (batch_size, 1)
        
        return out


### Model Results without Exogenous Features

### Model Results with Exogenous Features